# 🌌 ExoVision AI — Light Curve Acquisition & Exploratory Analysis

Welcome to the first notebook of **ExoVision AI**. This notebook introduces the fundamental concepts of astronomical photometry, light curve data loading, basic statistics, and visualization techniques for planetary transit detection.

## 1. Environment & Library Setup
We import core astronomical data processing libraries: `numpy`, `matplotlib`, `astropy`, `lightkurve`, and our custom ExoVision utility `ai.utils.lightcurve_loader`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import astropy
import lightkurve as lk

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai.utils.lightcurve_loader import load_lightcurve_fits, validate_lightcurve

print(f"Astropy Version: {astropy.__version__}")
print(f"Lightkurve Version: {lk.__version__}")

## 2. Load Astronomical Light Curve
We load a sample FITS light curve file from the `data/raw/kepler/` dataset using `load_lightcurve_fits()`.

In [ ]:
sample_path = PROJECT_ROOT / "data" / "raw" / "kepler" / "sample_kepler_kepler8.fits"
data = load_lightcurve_fits(sample_path)
validate_lightcurve(data)

time = data["time"]
flux = data["flux"]
flux_err = data["flux_error"]
quality = data["quality"]

## 3. Light Curve Statistics & Summary
Understanding basic statistical properties of a photometric time series helps characterize the host star and detect anomalies.

In [ ]:
num_observations = len(time)
duration_days = time[-1] - time[0]
missing_values = np.isnan(flux).sum()
mean_flux = np.mean(flux)
median_flux = np.median(flux)
std_flux = np.std(flux)
min_flux = np.min(flux)
max_flux = np.max(flux)

print("=== Light Curve Summary Statistics ===")
print(f"Number of Observations : {num_observations}")
print(f"Observation Duration   : {duration_days:.2f} days")
print(f"Missing Values (NaNs)  : {missing_values}")
print(f"Mean Flux              : {mean_flux:.2f} e-/s")
print(f"Median Flux            : {median_flux:.2f} e-/s")
print(f"Flux Std Dev           : {std_flux:.4f} e-/s")
print(f"Min Flux               : {min_flux:.2f} e-/s")
print(f"Max Flux               : {max_flux:.2f} e-/s")

## 4. Visualizations

### What is Stellar Flux & Brightness?
Stellar **flux** measures the total radiative photon energy arriving from a star per unit time per unit aperture area. When a transiting exoplanet passes directly in front of its parent star relative to the telescope observer, it blocks a portion of the star's light. This produces a periodic drop in measured flux known as a **transit dip**.

Below we generate three diagnostic plots:
1. **Raw Light Curve**: Unnormalized flux over time.
2. **Normalized Light Curve**: Flux divided by median flux.
3. **Flux Histogram**: Distribution of observed flux values.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 12))
fig.patch.set_facecolor('#0b0f19')

# Styling helper
for ax in axes:
    ax.set_facecolor('#111827')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    ax.grid(True, linestyle='--', alpha=0.3, color='#374151')

# Plot 1: Raw Light Curve
axes[0].plot(time, flux, color='#38bdf8', linewidth=1, label='Raw SAP/PDCSAP Flux')
axes[0].set_title('Plot 1: Raw Light Curve (Time vs Flux)', fontsize=14, pad=10)
axes[0].set_xlabel('Time (BJD - 2454833.0)', fontsize=11)
axes[0].set_ylabel('Flux (e-/s)', fontsize=11)
axes[0].legend(loc='upper right', facecolor='#1f2937', edgecolor='none', labelcolor='white')

# Plot 2: Normalized Light Curve
norm_flux = flux / median_flux
axes[1].plot(time, norm_flux, color='#8b5cf6', linewidth=1, label='Normalized Flux (Relative Brightness)')
axes[1].axhline(1.0, color='#ef4444', linestyle=':', alpha=0.7, label='Baseline (1.0)')
axes[1].set_title('Plot 2: Normalized Light Curve', fontsize=14, pad=10)
axes[1].set_xlabel('Time (BJD)', fontsize=11)
axes[1].set_ylabel('Relative Flux', fontsize=11)
axes[1].legend(loc='upper right', facecolor='#1f2937', edgecolor='none', labelcolor='white')

# Plot 3: Flux Histogram
axes[2].hist(norm_flux, bins=50, color='#10b981', edgecolor='#064e3b', alpha=0.85)
axes[2].set_title('Plot 3: Relative Flux Distribution Histogram', fontsize=14, pad=10)
axes[2].set_xlabel('Relative Flux', fontsize=11)
axes[2].set_ylabel('Observation Frequency', fontsize=11)

plt.tight_layout()
plt.show()

## 5. Scientific Summary & Noise Sources

### Sources of Noise in Astronomical Photometry
1. **Instrumental Systematic Artifacts**: Spacecraft pointing drifts, thermal fluctuations, and momentum dumps create non-astrophysical trends.
2. **Stellar Variability & Starspots**: Rotating starspots and flares produce brightness variations that can mask or mimic planet transits.
3. **Crowded Field Contamination**: Background stars spilling into the aperture dilute transit depths.
4. **Cosmic Rays**: High-energy particle impacts produce single-frame flux spikes.